# Classification

In [74]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
import joblib

pd.set_option('display.max_columns', None)

In [75]:
import warnings
from sklearn.exceptions import ConvergenceWarning, FitFailedWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=FitFailedWarning)


### 1. Caricamento dati

In [76]:
df_train = pd.read_csv('df_train.csv')
df_test = pd.read_csv('df_test.csv')

In [77]:
df_train.shape

(574, 302)

In [78]:
df_test.shape

(384, 302)

In [79]:
model_path = '/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/'

In [80]:
df_train['target'].value_counts()

target
2    401
1    119
0     54
Name: count, dtype: int64

In [81]:
## Creiamo un dizionario dove inserire i risultati della validazione
evaluation_results = {}

#### Selezione anni di interesse

In [82]:
cols_to_drop = df_train.filter(regex=r"2020|2021|2022").columns.tolist()

In [83]:
df_train = df_train.drop(cols_to_drop, axis = 1)

In [84]:
df_test = df_test.drop(cols_to_drop, axis = 1)

In [85]:
df_train.shape

(574, 275)

### 2. Divisione in train e validation

In [86]:
y = df_train['target']
X = df_train.drop('target', axis = 1)

In [87]:
X

,works_count_2023,cited_by_count_2023,tot_dipendenti_per_anno_2023,studiosi_usciti_2023,studiosi_entrati_2023,2023_Health Sciences,2023_Life Sciences,2023_Social Sciences,2023_Physical Sciences,role_funder,role_institution,role_publisher,regione_Abruzzo,regione_Aosta Valley,regione_Apulia,regione_Basilicate,regione_Calabria,regione_Campania,regione_Emilia-Romagna,regione_Friuli Venezia Giulia,regione_Lazio,regione_Liguria,regione_Lombardy,regione_Molise,regione_Piedmont,regione_Sardinia,regione_Sicily,regione_The Marches,regione_Trentino-Alto Adige,regione_Tuscany,regione_Umbria,regione_Veneto,macroarea_Centro,macroarea_Isole,macroarea_Nord,macroarea_Sud,type_archive,type_company,type_education,type_facility,type_government,type_healthcare,type_nonprofit,type_other,city_Abano Terme,city_Abbiategrasso,city_Acquaviva delle Fonti,city_Agrate Brianza,city_Albano Laziale,city_Alessandria,city_Ancona,city_Andria,city_Aosta,city_Apiro,city_Arezzo,city_Ariano Irpino,city_Arnesano,city_Asti,city_Avellino,city_Aviano,city_Barga,city_Bari,city_Basiglio,city_Bassano del Grappa,city_Benevento,city_Bergamo,city_Biella,city_Bollate,city_Bologna,city_Bolzano,city_Bosisio Parini,city_Bra,city_Brescia,city_Bresso,city_Brindisi,city_Bruneck,city_Buccino,city_Busto Arsizio,city_Buttrio,city_Cagliari,city_Calenzano,city_Camaiore,city_Camerino,city_Campobasso,city_Candiolo,city_Capua,city_Carpi,city_Carrara,city_Casale Monferrato,city_Casalecchio di Reno,city_Casamassima,city_Cascina,city_Caserta,city_Cassino,city_Castel Gandolfo,city_Castelfranco Veneto,city_Castellana Grotte,city_Castellanza,city_Catania,city_Catanzaro,city_Cattolica,city_Cefalù,city_Cernusco sul Naviglio,city_Cesena,city_Chieri,city_Chieti,city_Città Sant'Angelo,city_Città di Castello,city_Civita Castellana,city_Civitanova Marche,city_Cologno Monzese,city_Como,city_Cosenza,city_Cotignola,city_Crema,city_Cremona,city_Crevalcore,city_Crotone,city_Cuneo,city_Desio,city_Empoli,city_Enna,city_Faenza,city_Ferrara,city_Fisciano,city_Florence,city_Foggia,city_Foligno,city_Forlì,city_Formello,city_Francavilla Fontana,city_Frascati,city_Frosinone,city_Garda,city_Genoa,city_Gorizia,city_Gravedona,city_Grosseto,city_Grugliasco,city_Imola,city_Ispra,city_Ivrea,city_La Spezia,city_Lainate,city_Lanciano,city_Latina,city_Lavagna,city_Lecce,city_Lecco,city_Legnaro,city_Livorno,city_Lodi,city_Longastrino,city_Lucca,city_L’Aquila,city_Macerata,city_Maddaloni,city_Mangone,city_Mantova,city_Maranello,city_Marostica,city_Matera,city_Meldola,city_Meran,city_Messina,city_Mestre,city_Milan,city_Mirano,city_Modena,city_Molise,city_Moncalieri,city_Monfalcone,city_Monserrato,city_Monterotondo,city_Monza,city_Mozzagrogna,city_Naples,city_Narni,city_Negrar,city_Nerviano,city_Novara,city_Novedrate,city_Nus,city_Orbassano,city_Oristano,city_Ornavasso,city_Padova,city_Padua,city_Palermo,city_Parabiago,city_Parma,city_Pavia,city_Pavullo nel Frignano,city_Perugia,city_Pesaro,city_Pescara,city_Piacenza,city_Pietra Ligure,city_Pinerolo,city_Pino Torinese,city_Pisa,city_Pistoia,city_Pomezia,city_Pontecagnano,city_Pontedera,city_Porcia,city_Pordenone,city_Portici,city_Potenza,city_Povo,city_Pozzilli,city_Pozzuoli,city_Prato,city_Pula,city_Ragusa,city_Ravenna,city_Reggio Calabria,city_Reggio Emilia,city_Rende,city_Rimini,city_Rionero in Vulture,city_Rivoli,city_Rome,city_Rovereto,city_Rovigo,city_Rozzano,city_Salerno,city_San Donato Milanese,city_San Donà di Piave,city_San Giovanni Rotondo,city_San Martino Siccomario,city_San Michele all'Adige,city_Sanremo,city_Sansepolcro,city_Saronno,city_Sassari,city_Sasso Marconi,city_Savona,city_Scorrano,city_Segrate,city_Seriate,city_Sesto Fiorentino,city_Sesto San Giovanni,city_Siena,city_Sondalo,city_Sondrio,city_Spoleto,city_Tempio Pausania,city_Teramo,city_Terni,city_Tirrenia,city_Tito,city_Trento,city_Treviglio,city_Treviso,city_Trezzano Rosa,city_Trieste,city_Troina,city_Turin,city_Udine,city_Urbino,city_Valenzano,city_Varese,city_Venice,city_Verbania,city_Vercelli,city_Ve

In [88]:
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

In [89]:
df_train.columns

Index(['works_count_2023', 'cited_by_count_2023',
       'tot_dipendenti_per_anno_2023', 'studiosi_usciti_2023',
       'studiosi_entrati_2023', '2023_Health Sciences', '2023_Life Sciences',
       '2023_Social Sciences', '2023_Physical Sciences', 'role_funder',
       ...
       'city_Vercelli', 'city_Verona', 'city_Vicenza', 'city_Vietri sul Mare',
       'city_Villa Guardia', 'city_Vimercate', 'city_Viterbo',
       'city_Vittorio Veneto', 'city_Vizzolo Predabissi', 'target'],
      dtype='object', length=275)

### 3. Scalatura dei dati

In [90]:
cols_to_scale = df_train.filter(regex=r"\d{4}").columns.tolist()
scaler = StandardScaler()
x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_val[cols_to_scale] = scaler.transform(x_val[cols_to_scale])
df_test[cols_to_scale] = scaler.transform(df_test[cols_to_scale])

In [91]:
x_train.shape

(344, 274)

In [92]:
cv = 5

### 4. Fase di validazione
- KNN
- Logistic Regression
- SVM
- Gradient Boosting
- Random Forest
- Decision Tree
- Gaussian NB

#### KNN

n_neighbors=5, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None

In [93]:
knn_params = {
    'n_neighbors' : [3, 5, 10],
    'weights' : ['uniform', 'distance'],
    'algorithm' : ['auto', 'brute'],
    'leaf_size' : [10, 30, 50],
    'p': [1, 2, 5],
    'metric' : ['minkowski']
}

knn_grid = GridSearchCV(KNeighborsClassifier(),
                        knn_params,
                        cv = cv,
                        scoring = 'f1_macro',
                        verbose = 1)

In [94]:
knn_grid.fit(x_train, y_train)
print(knn_grid.best_params_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
{'algorithm': 'auto', 'leaf_size': 10, 'metric': 'minkowski', 'n_neighbors': 3, 'p': 5, 'weights': 'uniform'}


In [95]:
# Allenamento e performance sul training set
knn_model = KNeighborsClassifier(**knn_grid.best_params_)
knn_model.fit(x_train, y_train)
y_pred_train_knn = knn_model.predict(x_train)

print(f'Accuracy di KNN sul training: {accuracy_score(y_train, y_pred_train_knn)}')
print(f'F1 macro di KNN sul training: {f1_score(y_train, y_pred_train_knn, average = 'macro')}')

Accuracy di KNN sul training: 0.7906976744186046
F1 macro di KNN sul training: 0.665087933080736


In [96]:
# Performance sul validation set
y_pred_val_knn = knn_model.predict(x_val)
accuracy_knn = accuracy_score(y_val, y_pred_val_knn)
f1_knn = f1_score(y_val, y_pred_val_knn, average='macro')

print(f'Accuracy KNN sul validation: {accuracy_knn}')
print(f'F1 Macro del KNN sul validation set: {f1_knn}')

print('Classification report per KNN')
print(classification_report(y_val, y_pred_val_knn))

Accuracy KNN sul validation: 0.6391304347826087
F1 Macro del KNN sul validation set: 0.39765812281025265
Classification report per KNN
              precision    recall  f1-score   support

           0       0.17      0.19      0.18        21
           1       0.40      0.17      0.24        48
           2       0.72      0.84      0.78       161

    accuracy                           0.64       230
   macro avg       0.43      0.40      0.40       230
weighted avg       0.60      0.64      0.61       230



In [97]:
# Salvataggio risultati
evaluation_results['KNN'] = {
    'accuracy' : "{:.2f}".format(accuracy_knn),
    'f1' : "{:.2f}".format(f1_knn)
}

### Logistic Reg

In [98]:
log_reg_params = [
    {
        'solver': ['lbfgs', 'newton-cg'],
        'C': [0.01, 0.1, 1],
        'class_weight': ['balanced'],
        'max_iter': [300, 1000],
        'penalty': ['l2']
    },
    {
        'solver': ['sag', 'saga'],
        'C': [0.01, 0.1, 1],
        'class_weight': ['balanced'],
        'max_iter': [1000, 5000],
        'penalty': ['l2']
    }
]


grid_log_reg = GridSearchCV(LogisticRegression(),
                            log_reg_params, 
                            cv = 5, 
                            scoring = 'f1_macro',
                            verbose=1,
                            )

In [99]:
grid_log_reg.fit(x_train, y_train)
print(grid_log_reg.best_params_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
{'C': 1, 'class_weight': 'balanced', 'max_iter': 300, 'penalty': 'l2', 'solver': 'lbfgs'}


In [100]:
# Allenamento e performance sul training
log_reg_model = LogisticRegression(**grid_log_reg.best_params_, random_state=42)
log_reg_model.fit(x_train, y_train)
y_pred_train = log_reg_model.predict(x_train)
print(f'F1 Score della Logistic Regression sul training set: {f1_score(y_train, y_pred_train, average="weighted")}')
print(f'Accuracy della Logistic Regression sul training set: {accuracy_score(y_train, y_pred_train)}')

F1 Score della Logistic Regression sul training set: 0.7550678643784107
Accuracy della Logistic Regression sul training set: 0.7412790697674418


In [101]:
joblib.dump(
    log_reg_model,
    model_path + 'log_reg_due_city'+ '.pkl'
)

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/log_reg_due_city.pkl']

In [102]:
#log_reg_model = joblib.load('/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/_log_reg_due_anni.pkl')

In [103]:
# Performance sul validation
y_pred_log_reg = log_reg_model.predict(x_val)
lr_accuracy = accuracy_score(y_val, y_pred_log_reg)
lr_f1 = f1_score(y_val, y_pred_log_reg, average="macro")


print('Classification Report per Logistic Regression')
print(classification_report(y_val, y_pred_log_reg))
print(f'Accuracy della Logistic Regression: {lr_accuracy}')
print(f'F1 Score della Logistic Regression: {lr_f1}')

Classification Report per Logistic Regression
              precision    recall  f1-score   support

           0       0.44      0.57      0.50        21
           1       0.35      0.58      0.43        48
           2       0.82      0.62      0.71       161

    accuracy                           0.61       230
   macro avg       0.54      0.59      0.55       230
weighted avg       0.69      0.61      0.63       230

Accuracy della Logistic Regression: 0.6086956521739131
F1 Score della Logistic Regression: 0.5469407693501703


In [104]:
# Salvataggio risultati
evaluation_results['Logistic Regression'] = {
    'accuracy' : "{:.2f}".format(lr_accuracy),
    'f1' : "{:.2f}".format(lr_f1)
}

### SVM

In [105]:
svm_params = {
    'kernel' : ['rbf', 'linear', 'poly', 'sigmoid'], 
    'degree' : [2,3,5],
    'C' : [0.001, 0.01, 0.1, 1],
    'gamma': ['scale', 'auto'], 
    'class_weight' : ['balanced'],
    'max_iter' : [100, 300, 500, 1000]
}

In [106]:
svm_grid = GridSearchCV(SVC(), 
                        svm_params,
                        cv = 5,
                        scoring = 'f1_macro',
                        verbose = 1)

In [107]:
svm_grid.fit(x_train, y_train)
print(svm_grid.best_params_)

Fitting 5 folds for each of 384 candidates, totalling 1920 fits
{'C': 1, 'class_weight': 'balanced', 'degree': 2, 'gamma': 'scale', 'kernel': 'linear', 'max_iter': 1000}


In [108]:
# Allenamento e performance sul training set
svm_model = SVC(**svm_grid.best_params_, probability=True, random_state=42)
svm_model.fit(x_train, y_train)

y_pred_train = svm_model.predict(x_train)
print(f'Accuracy di SVM: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score di SVM: {f1_score(y_train, y_pred_train, average="macro")}')

Accuracy di SVM: 0.75
F1 Score di SVM: 0.7121972160511715


In [109]:
joblib.dump(
    svm_model,
    model_path + '_svm_tesi_uno_city'+ '.pkl'
)
#svm_model = joblib.load('/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_svm_tesi_uno_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_svm_tesi_uno_city.pkl']

In [110]:
# Performance sul validation set
y_pred_svm = svm_model.predict(x_val)
svm_accuracy = accuracy_score(y_val, y_pred_svm)
svm_f1 = f1_score(y_val, y_pred_svm, average="macro")


print('Classification Report per SVClassifier')
print(classification_report(y_val, y_pred_svm))
print(f'Accuracy di SVM: {svm_accuracy}')
print(f'F1 Score di SVM: {svm_f1}')

Classification Report per SVClassifier
              precision    recall  f1-score   support

           0       0.37      0.62      0.46        21
           1       0.32      0.58      0.41        48
           2       0.81      0.54      0.65       161

    accuracy                           0.56       230
   macro avg       0.50      0.58      0.51       230
weighted avg       0.67      0.56      0.58       230

Accuracy di SVM: 0.5565217391304348
F1 Score di SVM: 0.5084347171704503


In [111]:
# Salvataggio risultati
evaluation_results['SVM'] = {
    'accuracy' : "{:.2f}".format(svm_accuracy),
    'f1' : "{:.2f}".format(svm_f1)
}

### Gradient Boosting

In [112]:
gboost_params = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],           
    'min_samples_leaf': [10, 20, 30],
    'subsample': [0.6, 0.8, 1.0],    
    'ccp_alpha': [0.01, 0.1, 0.5]
}

gboost_grid = GridSearchCV(GradientBoostingClassifier(),
                           gboost_params, 
                           cv = 5, 
                           scoring='f1_macro',
                           n_jobs=-1,
                           verbose=1)

gboost_grid.fit(x_train, y_train)
print(gboost_grid.best_params_)

Fitting 5 folds for each of 729 candidates, totalling 3645 fits
{'ccp_alpha': 0.01, 'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 10, 'n_estimators': 200, 'subsample': 0.6}


In [113]:
# Allenamento e performance sul training set
gb_model = GradientBoostingClassifier(**gboost_grid.best_params_, random_state=42)
gb_model.fit(x_train, y_train)
y_train_pred = gb_model.predict(x_train)


print("Train Accuracy di Gradient Boosting:", accuracy_score(y_train, y_train_pred))
print(f'F1 Score di Gradient Boosting: {f1_score(y_train, y_pred_train, average="macro")}')

Train Accuracy di Gradient Boosting: 0.6976744186046512
F1 Score di Gradient Boosting: 0.7121972160511715


In [114]:
joblib.dump(
    gb_model,
    model_path + '_gb_tesi_uno_city'+ '.pkl'
)
#gb_model = joblib.load(f'{model_path}_gb_tesi.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_gb_tesi_uno_city.pkl']

In [115]:
# Performance on the validation set
y_pred_gb = gb_model.predict(x_val)
gb_accuracy = accuracy_score(y_val, y_pred_gb)
gb_f1 = f1_score(y_val, y_pred_gb, average="macro")

print('Classification Report per Gradient Boosting Classifier')
print(classification_report(y_val, y_pred_gb))
print(f'Accuracy del GBClassifier: {gb_accuracy}')
print(f'F1 Score del GBClassifier: {gb_f1}')

Classification Report per Gradient Boosting Classifier
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        21
           1       1.00      0.04      0.08        48
           2       0.71      1.00      0.83       161

    accuracy                           0.71       230
   macro avg       0.57      0.35      0.30       230
weighted avg       0.70      0.71      0.60       230

Accuracy del GBClassifier: 0.7086956521739131
F1 Score del GBClassifier: 0.3025878320479863


/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [116]:
# Salvataggio performance
evaluation_results['Gradient Boosting'] = {
    'accuracy' : "{:.2f}".format(gb_accuracy),
    'f1' : "{:.2f}".format(gb_f1)
}

### Random Forest

In [117]:
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],         
    'min_samples_split': [5, 10, 20], 
    'min_samples_leaf': [4, 8, 16],   
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}

rf_grid = GridSearchCV(RandomForestClassifier(),
                       rf_params, 
                       cv = 5,
                       verbose = 1,
                       n_jobs = -1, 
                       scoring = 'f1_macro')

In [118]:
rf_grid.fit(x_train, y_train)
print(rf_grid.best_params_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
{'class_weight': 'balanced', 'max_depth': 3, 'max_features': 'log2', 'min_samples_leaf': 16, 'min_samples_split': 5, 'n_estimators': 200}


In [119]:
# Allenamento e performance sul training set
rf_model = RandomForestClassifier(**rf_grid.best_params_, random_state=42)
rf_model.fit(x_train, y_train)
y_pred_train = rf_model.predict(x_train)

print(f'Accuracy di Random Forest: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score di Random Forest: {f1_score(y_train, y_pred_train, average="macro")}')

Accuracy di Random Forest: 0.5726744186046512
F1 Score di Random Forest: 0.495750087365578


In [120]:
joblib.dump(
    rf_model,
    model_path + '_rf_tesi_uno_city'+ '.pkl'
)
#rf_model = joblib.load('/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_rf_tesi_uno_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_rf_tesi_uno_city.pkl']

In [121]:
# Performance sul validation set
y_pred_rf = rf_model.predict(x_val)
rf_accuracy = accuracy_score(y_val, y_pred_rf)
rf_f1 = f1_score(y_val, y_pred_rf, average="macro")

print('Classification Report per Random Forest')
print(classification_report(y_val, y_pred_rf))
print(f'Accuracy del Random Forest: {rf_accuracy}')
print(f'F1 Score del Random Forest: {rf_f1}')

Classification Report per Random Forest
              precision    recall  f1-score   support

           0       0.21      0.76      0.33        21
           1       0.44      0.33      0.38        48
           2       0.86      0.63      0.73       161

    accuracy                           0.58       230
   macro avg       0.50      0.58      0.48       230
weighted avg       0.71      0.58      0.62       230

Accuracy del Random Forest: 0.5826086956521739
F1 Score del Random Forest: 0.48095238095238085


In [122]:
# Salvataggio risultati
evaluation_results['Random Forest'] = {
    'accuracy' : "{:.2f}".format(rf_accuracy),
    'f1' : "{:.2f}".format(rf_f1)
}

### Decision Tree

In [123]:
dt_params = {
    'criterion' : ['gini', 'entropy', 'log_loss'],
    'max_depth' : [2, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced'],
    'max_features' : ['sqrt', 'log2']
}

dt_grid = GridSearchCV(DecisionTreeClassifier(), 
                       dt_params,
                       cv = 5, 
                       verbose = 1,
                       n_jobs = -1,
                       scoring = 'f1_macro')

In [124]:
dt_grid.fit(x_train, y_train)
print(dt_grid.best_params_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2}


In [125]:
# Allenamento e performance sul training set
dt_model = DecisionTreeClassifier(**dt_grid.best_params_, random_state=42)
dt_model.fit(x_train, y_train)

y_pred_train = dt_model.predict(x_train)
print(f'Accuracy del Decision Tree: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Macro del Decision Tree: {f1_score(y_train, y_pred_train, average = 'macro')}')

Accuracy del Decision Tree: 0.311046511627907
F1 Macro del Decision Tree: 0.3146466039234543


In [126]:
joblib.dump(
    dt_model,
    model_path + '_dt_tesi_uno_city'+ '.pkl'
)
#dt_model = joblib.load('/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_dt_tesi_uno_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_dt_tesi_uno_city.pkl']

In [127]:
# Performance sul validation set
y_pred_dt = dt_model.predict(x_val)
dt_accuracy = accuracy_score(y_val, y_pred_dt)
dt_f1 = f1_score(y_val, y_pred_dt, average="macro")

print('Classification Report per Decision Tree')
print(classification_report(y_val, y_pred_dt))
print(f'Accuracy del Decision Tree: {dt_accuracy}')
print(f'F1 Macro del Decision Tree: {dt_f1}')

Classification Report per Decision Tree
              precision    recall  f1-score   support

           0       0.15      0.67      0.25        21
           1       0.28      0.62      0.38        48
           2       0.86      0.16      0.26       161

    accuracy                           0.30       230
   macro avg       0.43      0.48      0.30       230
weighted avg       0.67      0.30      0.29       230

Accuracy del Decision Tree: 0.3
F1 Macro del Decision Tree: 0.29770370348395087


In [128]:
evaluation_results['Decision Tree'] = {
    'accuracy' : "{:.2f}".format(dt_accuracy),
    'f1' : "{:.2f}".format(dt_f1)
}

### XGBoost

In [129]:
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 1],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 5]
}

xgb_grid = GridSearchCV(XGBClassifier(), 
                       xgb_params,
                       cv = 5, 
                       verbose = 1,
                       n_jobs = -1,
                       scoring = 'f1_macro')

In [130]:
xgb_grid.fit(x_train, y_train)
print(xgb_grid.best_params_)

Fitting 5 folds for each of 864 candidates, totalling 4320 fits
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 1.0}


In [131]:
# Allenamento e performance sul training set
xgb_model = XGBClassifier(**xgb_grid.best_params_, random_state = 42)
xgb_model.fit(x_train, y_train)

y_pred_xgb_train = xgb_model.predict(x_train)
print(f'Accuracy di XGBoost: {accuracy_score(y_train, y_pred_xgb_train)}')
print(f'F1 Score di XGBoost: {f1_score(y_train, y_pred_xgb_train, average="macro")}')

Accuracy di XGBoost: 1.0
F1 Score di XGBoost: 1.0


In [132]:
joblib.dump(
    xgb_model,
    model_path + '_xgb_tesi_uno_city'+ '.pkl'
)
#xgb_model = joblib.load(f'{model_path}_xgb_tesi.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli ad un anno con city/_xgb_tesi_uno_city.pkl']

In [133]:
# Performance sul validation set
y_pred_xgb = xgb_model.predict(x_val)
xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb, average="macro")

print('Classification Report per XGBoost')
print(classification_report(y_val, y_pred_xgb))
print(f'Accuracy di XGBoost: {xgb_accuracy}')
print(f'F1 Score di XGBoost: {xgb_f1}')

Classification Report per XGBoost
              precision    recall  f1-score   support

           0       0.50      0.29      0.36        21
           1       0.48      0.27      0.35        48
           2       0.76      0.90      0.82       161

    accuracy                           0.71       230
   macro avg       0.58      0.49      0.51       230
weighted avg       0.68      0.71      0.68       230

Accuracy di XGBoost: 0.7130434782608696
F1 Score di XGBoost: 0.5113888888888889


In [134]:
# Salvataggio risultati
evaluation_results['XGBoost'] = {
    'accuracy' : "{:.2f}".format(xgb_accuracy),
    'f1' : "{:.2f}".format(xgb_f1)
}

### Risultati validazione e ri allenamento

In [135]:
validation_df =  pd.DataFrame({'Model': list(evaluation_results.keys()),
                               'Results' : list(evaluation_results.values())})

In [136]:
validation_df

,Model,Results
0,KNN,"{'accuracy': '0.64', 'f1': '0.40'}"
1,Logistic Regression,"{'accuracy': '0.61', 'f1': '0.55'}"
2,SVM,"{'accuracy': '0.56', 'f1': '0.51'}"
3,Gradient Boosting,"{'accuracy': '0.71', 'f1': '0.30'}"
4,Random Forest,"{'accuracy': '0.58', 'f1': '0.48'}"
5,Decision Tree,"{'accuracy': '0.30', 'f1': '0.30'}"
6,XGBoost,"{'accuracy': '0.71', 'f1': '0.51'}"


In [137]:
## Unificazione training set
x_train_final = pd.concat([x_train, x_val])
y_train_final  = pd.concat([y_train, y_val])

In [138]:
## Ri allenamento dei modelli
### KNN ###
knn_model_final = KNeighborsClassifier(**knn_grid.best_params_)
knn_model_final.fit(x_train_final, y_train_final)

### LOGISTIC REGRESSION ###
log_reg_model_final = LogisticRegression(**grid_log_reg.best_params_, random_state=42)
log_reg_model_final.fit(x_train_final, y_train_final)

### SVM ###
svm_model_final = SVC(**svm_grid.best_params_, random_state=42)
svm_model_final.fit(x_train_final, y_train_final)

### RANDOM FOREST ###
rf_model_final = RandomForestClassifier(**rf_grid.best_params_, random_state=42)
rf_model_final.fit(x_train_final, y_train_final)

### XGBOOST ###
xgb_model_final = XGBClassifier(**xgb_grid.best_params_, random_state = 42)
xgb_model_final.fit(x_train_final, y_train_final)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

## Evaluation

In [139]:
y_test = df_test['target']
x_test = df_test.drop('target', axis=1)

In [141]:
lab_encoder = joblib.load('/Users/laviniarotellini/Desktop/Laurea/Tesi/Costruzione e Analisi DF/3. Data Understanding and Prep/label_encoder.pkl')

In [142]:
# Tutti i modelli con i rispettivi predittori
trained_models = {
    'KNN': (knn_model_final, y_pred_val_knn),
    'Logistic Regression': (log_reg_model_final, y_pred_log_reg),
    'SVM': (svm_model_final, y_pred_svm),
    'Random Forest': (rf_model_final, y_pred_rf),
    'XGBoost': (xgb_model_final, y_pred_xgb),
}

test_results = []

for name, (model, _) in trained_models.items():
    y_test_pred = model.predict(x_test)
    
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1_weighted = f1_score(y_test, y_test_pred, average='weighted')
    test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
    
    test_results.append({
        'Model': name,
        'Test Accuracy': test_accuracy,
        'Test F1 Weighted': test_f1_weighted,
        'Test F1 Macro': test_f1_macro,
    })
    
    print(f'\n{name}')
    print(classification_report(y_test, y_test_pred, 
                                  target_names=lab_encoder.classes_))
    print('*' * 50)

test_results_df = pd.DataFrame(test_results).sort_values(
    by='Test F1 Macro', ascending=False
)
print(test_results_df)


KNN
              precision    recall  f1-score   support

      inflow       0.29      0.26      0.27        38
     outflow       0.35      0.21      0.26        90
      stable       0.71      0.82      0.76       256

    accuracy                           0.62       384
   macro avg       0.45      0.43      0.43       384
weighted avg       0.59      0.62      0.60       384

**************************************************

Logistic Regression
              precision    recall  f1-score   support

      inflow       0.24      0.45      0.31        38
     outflow       0.40      0.48      0.43        90
      stable       0.78      0.62      0.69       256

    accuracy                           0.57       384
   macro avg       0.47      0.52      0.48       384
weighted avg       0.64      0.57      0.59       384

**************************************************

SVM
              precision    recall  f1-score   support

      inflow       0.24      0.55      0.34       